### 1. Import Necessary Libraries
We start by importing all the required Python libraries for data manipulation, visualization, and machine learning.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from imblearn.over_sampling import RandomOverSampler


### 2. Load the Dataset
Loading the IPL match data from the CSV file into a Pandas DataFrame.

In [ ]:
data = pd.read_csv(r"C:\Users\rohit\Desktop\ML Presentation\matches.csv")

### 3. Data Preview
Display the first 5 rows of the dataset to understand its structure.

In [ ]:
data.head()

### 4. Statistical Summary
View basic statistical details like mean, standard deviation, and percentiles for numerical columns.

In [ ]:
data.describe()

### 5. Check for Missing Values
Identify columns with missing (null) values so we know what needs cleaning.

In [ ]:
data.isnull().sum()

### 6. Handle Missing Data
Drop columns or rows with too many null values. Here we remove the last column (which had many nulls) and drop any remaining rows with missing data.

In [ ]:
data = data.iloc[:,:-1]
data.dropna(inplace=True)

### 7. Unique Teams
View the unique team names present in the dataset.

In [ ]:
data["team1"].unique()

### 8. Standardize Team Names
Some teams changed their names over the years (e.g., Delhi Daredevils to Delhi Capitals). We replace old names with current ones to maintain consistency.

In [ ]:
#for Delhi Capitalsdata['team1']=data['team1'].str.replace('Delhi Daredevils','Delhi Capitals')data['team2']=data['team2'].str.replace('Delhi Daredevils','Delhi Capitals')data['winner']=data['winner'].str.replace('Delhi Daredevils','Delhi Capitals')#for sunrisers Hyderabaddata['team1']=data['team1'].str.replace('Deccan Chargers','Sunrisers Hyderabad')data['team2']=data['team2'].str.replace('Deccan Chargers','Sunrisers Hyderabad')data['winner']=data['winner'].str.replace('Deccan Chargers','Sunrisers Hyderabad')

### 9. Visualization: Matches Won by Each Team
A bar chart showing the total number of matches won by each team in the IPL.

In [ ]:
plt.figure(figsize=(12,7))
sns.set_style("darkgrid")

sns.countplot(
    y='winner',
    data=data,
    order=data['winner'].value_counts().index,
    palette='viridis'
)

plt.title('IPL Wins by Team', fontsize=16, fontweight='bold')
plt.xlabel('Wins')
plt.ylabel('Team')

plt.show()

### 10. Visualization: Top Venues
Displaying the stadiums that have hosted the most IPL matches.

In [ ]:
plt.figure(figsize = (10,6))
sns.countplot(y = 'venue',data = data,order = data['venue'].value_counts().iloc[:10].index)
plt.xlabel('No of matches',fontsize=12)
plt.ylabel('Venue',fontsize=12)
plt.title('Total Number of matches played in different stadium')

### 11. Visualization: Toss Decisions
A plot to see how often captains choose to bat versus field after winning the toss.

In [ ]:
plt.figure(figsize=(10,6))
sns.set_style("whitegrid")

sns.countplot(
    x="toss_decision",
    data=data,
    palette="coolwarm"
)

plt.title('Toss Decision Distribution', fontsize=15, fontweight='bold')
plt.xlabel('Toss Decision')
plt.ylabel('Number of Matches')

plt.show()

### 12. Inspect Categorical Columns
Checking the unique values and their frequencies for categorical features.

In [ ]:
x = ["city", "toss_decision", "result", "dl_applied"]

for i in x:
    print("------------")
    print(data[i].unique())
    print(data[i].value_counts())

### 13. Drop Unnecessary Columns
Removing columns that don't help in predicting the match winner (like umpire names, date, and match ID).

In [ ]:
data.drop(
    ["id", "season", "city", "date",
     "player_of_match", "umpire1",
     "venue", "umpire2"],
    axis=1,
    inplace=True
)

### 14. Verify Columns
Checking the remaining columns after dropping.

In [ ]:
print(data.columns)

### 15. Double Check Dropped Columns
Ensuring all unnecessary columns are completely removed.

In [ ]:
drop_cols = ["id", "Season", "city", "date",
             "player_of_match", "umpire1",
             "venue", "umpire2"]

data.drop(columns=[col for col in drop_cols if col in data.columns], inplace=True)

### 16. Separate Features and Target
`X` contains all the input features (match details), while `y` contains the target variable we want to predict (the winner).

In [ ]:
X = data.drop(["winner"], axis=1)
y = data["winner"]

### 17. One-Hot Encoding
Machine learning models only understand numbers. We convert categorical text data (like team names) into binary columns (0s and 1s).

In [ ]:
X = pd.get_dummies(
    X,
    columns=["team1", "team2", "toss_winner", "toss_decision", "result"],
    drop_first=True   # avoids multicollinearity (dummy variable trap)
)

print("Encoded feature shape:", X.shape)
print("Sample columns:", list(X.columns[:6]), "...")

### 18. Encode Target Variable
Converting the target labels (team names) into numerical values using LabelEncoder (e.g., Team A = 0, Team B = 1).

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

### 19. Train-Test Split
Splitting the data so 80% is used to train the model, and 20% is kept hidden to test the model's accuracy later.

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, train_size = 0.8)

### 20. Initialize the Model
We use a Random Forest Classifier, which is a powerful algorithm that builds multiple decision trees to make accurate predictions.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

x_train = np.array(x_train)
y_train = np.array(y_train).ravel()

model = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=3,
    max_features='sqrt',
    random_state=42
)

### 21. Train the Model
Feeding the training data to the Random Forest so it can learn the patterns of who wins under what conditions.

In [ ]:
model.fit(x_train, y_train)

### 22. Make Predictions
Using the trained model to predict the winners for our hidden 20% test data.

In [ ]:
y_pred = model.predict(x_test)

### 23. Import Accuracy Metric

In [ ]:
from sklearn.metrics import accuracy_score

### 24. Evaluate Accuracy
Comparing our model's predictions to the actual test data answers to calculate an accuracy score.

In [ ]:
accuracy_score(y_pred, y_test)

### 25. Import Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

### 26. Calculate Confusion Matrix
A matrix showing how many correct and incorrect predictions the model made for each individual team.

In [ ]:
confusion_matrix(y_test, y_pred)

### 27. Visualize Confusion Matrix
Plotting the confusion matrix as a heatmap for much easier interpretation of the model's performance.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()